# Model and Evaluation (No Points Feature) for Approval Predict 

## Objective:

* Answer business requirement 2:
The client aims to offer a guide for potential applicants by identifying the influential factors that contribute to loan approval. These insights will be used to recommend specific improvements for applicants and guide them to increase their chances of having a loan approved.

Fit and evaluate classification models without `points` feature to predict loan approval outcomes. Classification will directly predict whether a loan is approved.

## Inputs
* outputs/datasets/collection/loan_approved.csv
* Instructions on which variables to use for data cleaning and feature engineering. They are found in each respective notebook.

## Outputs
* Train set (features and target)
* Test set (features and target)
* Machine learning pipeline including scaling & transformations
* Modeling pipeline
* Feature importance plot

## Additional Comments
* Test model performance without engineered points feature due to previous notebook identifying a 100% prediction accuracy. 
* Please see notebook 8 for more details regarding hyperparameter optimisation. 

## Add Imports

In [ ]:
import os
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from feature_engine.transformation import BoxCoxTransformer
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    make_scorer,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
sns.set(style="whitegrid")

## Change Working Directory

In [ ]:
current_dir = os.getcwd()
current_dir

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

In [ ]:
current_dir = os.getcwd()
current_dir

## Load data

The dataset is loaded with name and city removed.
The dataframe for the target `loan_approved` is converted to an iteger.
The loan_to_income feature is added.
The variable `no_point_features` is created for analysis.

In [ ]:
root = current_dir
file_path = (
    Path(root)
    / "outputs"
    / "datasets"
    / "collection"
    / "loan_approval.csv"
)

if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {file_path}"
    )

df = pd.read_csv(file_path).drop(['name', 'city'], axis=1)
df['loan_approved'] = df['loan_approved'].astype(int)
df['loan_to_income'] = df['loan_amount'] / df['income']

no_point_features = [
    "loan_to_income",
    "credit_score",
    "years_employed",
    "loan_amount",
    "income",
]
X = df[no_point_features].copy()
y = df['loan_approved'].copy()

X.head(3), y.head(3)

## Split into Train/Test sets

The dataset is split into a train (80%) and test set (20%). The dataset was trialed with 70/30 but did not add any added value to the findings.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print("Train/Test Shapes:")
X_train.shape, y_train.shape, X_test.shape, y_test.shape

Handle Class Imbalance

In notebook 2 it was identified that there was a minor imblanace of the target. For example, the data had 1121 rejections and 879 approvals. Therefore, we handled class imbalance using Synthetic Minority Oversampling Technique (SMOTE). This creates synthetic samples of the minority class. This is aimed to stop any bias when performing the machine learning.

The bar plot below shows an equal distribution of the target (1) loan approved and (0) loan rejected.

In [ ]:
smote = SMOTE(sampling_strategy='minority', random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("After SMOTE:", X_train_bal.shape, y_train_bal.shape)

y_train_bal.value_counts().plot(
    kind="bar",
    title="Balanced Loan Approved Distribution",
)

plt.show()

Build Classification ML pipeline

In [ ]:
def loan_approval_pipeline(model):
    """
    Create a machine learning pipeline that applies Box-Cox
    transformations, robust scaling, and a classification model for predicting
    loan approval.
    """
    pipeline = Pipeline([
        ('boxcox', BoxCoxTransformer(variables=['loan_to_income'])),
        ('scaler', RobustScaler()),
        ('model', model)
    ])
    return pipeline

In [ ]:
def PipelineClf(model):
    """
    Builds a pipeline for hyperparameter optimisation that includes
    feature transformations, robust scaling and a classifier model.
    """
    return Pipeline([
        ("boxcox", BoxCoxTransformer(variables=['loan_to_income'])),
        ("scaler", RobustScaler()),
        ("model", model),
    ])

## Grid search CV - Sklearn

This step identifies the best-performing model before detailed hyperparameter tuning.

In [ ]:
models_quick_search = {
    "LogisticRegression": LogisticRegression(random_state=42),
    "DecisionTreeClassifier": DecisionTreeClassifier(random_state=42),
    "RandomForestClassifier": RandomForestClassifier(random_state=42),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=42),
    "ExtraTreesClassifier": ExtraTreesClassifier(random_state=42),
    "AdaBoostClassifier": AdaBoostClassifier(random_state=42),
    "XGBClassifier": XGBClassifier(random_state=42)
}

params_quick_search = {
    "LogisticRegression": {},
    "DecisionTreeClassifier": {},
    "RandomForestClassifier": {},
    "GradientBoostingClassifier": {},
    "ExtraTreesClassifier": {},
    "AdaBoostClassifier": {},
    "XGBClassifier": {},
}

## Hyperparameter Optimisation

A custom class, HyperparameterOptimizationSearch, was created by the CodeInstitute to perform grid searches across selected parameter spaces for each model.

In [ ]:
class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")

            model = PipelineClf(self.models[key])
            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring, )
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)
        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]
        return df[columns], self.grid_searches


## Hyperparameter search

In [ ]:
search = HyperparameterOptimizationSearch(
    models=models_quick_search,
    params=params_quick_search
)

search.fit(
    X_train_bal,
    y_train_bal,
    scoring=make_scorer(recall_score, pos_label=1),
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [ ]:
results_summary, search_pipelines = search.score_summary(sort_by='mean_score')
display(results_summary)

In [ ]:
best_model = results_summary.iloc[0, 0]
pipeline_clf = search.grid_searches[best_model].best_estimator_
pipeline_clf

The following functions are used to evaluate the machine learning classifier by producing a confusion matrix and a classification report for both the training and test datasets.

In [ ]:
def confusion_matrix_and_report(X, y, pipeline, label_map):
    prediction = pipeline.predict(X)

    print('---  Confusion Matrix  ---')
    print(pd.DataFrame(
        confusion_matrix(y_true=prediction, y_pred=y),
        columns=[["Actual " + sub for sub in label_map]],
        index=[["Prediction " + sub for sub in label_map]]
    ))
    print("\n")

    print('---  Classification Report  ---')
    print(classification_report(y, prediction, target_names=label_map), "\n")


def clf_performance(X_train, y_train, X_test, y_test, pipeline, label_map):
    print("#### Train Set #### \n")
    confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

    print("#### Test Set ####\n")
    confusion_matrix_and_report(X_test, y_test, pipeline, label_map)

In [ ]:
print(f"Quick-search best model: {best_model}")

clf_performance(
    X_train_bal, y_train_bal,
    X_test, y_test,
    pipeline_clf,
    label_map=["Rejected", "Approved"]
)

We use a bar plot to show which features influence the model's predictions the most.

In [ ]:
top_features = X_train.columns

df_feature_importance = (pd.DataFrame(data={
    'Feature': top_features,
    'Importance': pipeline_clf['model'].feature_importances_})
    .sort_values(by='Importance', ascending=False)
)

df_feature_importance.plot(kind="bar", x="Feature", y="Importance")
plt.title("Feature Importance")
plt.show()
plt.tight_layout()
plt.show()

## Final Model

The final ML model was LogisticRegression, which after hyperparameter tuning (see note book 8 HyperParameterOptimisation) found recall of 0.87 and precision of 0.86 on the test set using the features `credit_score`, `income`, `loan_to_income`, `years_employed` and `loan_amout`. Points was removed as this did not require any hyperparameter tuning due to a prescion and accuray of 1 on all grid searches. Therefore, points would cause the model to be biased towards points score and not allow any weighting of the other features.

In [ ]:
model = {
    "LogisticRegression": LogisticRegression(
        random_state=42,
        max_iter=1000,
        solver="liblinear",
       )
    }

In [ ]:
lr_best = {
    "LogisticRegression": {
        'model__C': [0.1],
        'model__penalty': ['l2'],
        'model__class_weight': [None],
    }
}

In [ ]:
search = HyperparameterOptimizationSearch(models=model, params=lr_best)
search.fit(X_train_bal, y_train_bal,
           scoring=make_scorer(recall_score, pos_label=1),
           cv=5, n_jobs=-1)

In [ ]:
extensive_grid_search_summary, extensive_grid_search_pipelines = (
    search.score_summary(sort_by="mean_score")
)

best_model = extensive_grid_search_summary.iloc[0, 0]

best_parameters = extensive_grid_search_pipelines[best_model].best_params_

classification_pipeline = (
    extensive_grid_search_pipelines[best_model].best_estimator_
)

best_parameters

In [ ]:
pipeline_lr = Pipeline([
    ("boxcox", BoxCoxTransformer(variables=['loan_to_income'])),
    ("scaler", RobustScaler()),
    ("model", lr_best)
])

In [ ]:
clf_performance(X_train=X_train_bal, y_train=y_train_bal,
                X_test=X_test, y_test=y_test,
                pipeline=classification_pipeline,
                label_map=["Rejected", "Approved"])

In [ ]:
y_pred_train = classification_pipeline.predict(X_train)
y_pred_test = classification_pipeline.predict(X_test)

print("#### Train Set ####\n")
print(confusion_matrix(y_train, y_pred_train))
print(
    classification_report(
        y_train,
        y_pred_train,
        target_names=["Rejected", "Approved"],
    )
)

print("#### Test Set ####\n")
print(confusion_matrix(y_test, y_pred_test))
print(
    classification_report(
        y_test,
        y_pred_test,
        target_names=["Rejected", "Approved"],
    )
)

## Save Model

The following files will be saved:
* Train set (balanced)
* Test set
* ML pipeline (LogisticRegression for final ML model)
* Feature plot


In [ ]:
version = "v2"
file_path = f"outputs/ml_pipeline/approval_prediction/{version}"

try:
    os.makedirs(name=file_path)
except Exception as e:
    print(e)

### Train Set

Save train set.


In [ ]:
print(X_train_bal.shape)
X_train_bal.head(3)

In [ ]:
X_train_bal.to_csv(f"{file_path}/X_train.csv", index=False)
y_train_bal.to_csv(f"{file_path}/y_train.csv", index=False)

### Test Set

Save test set.


In [ ]:
print(X_test.shape)
X_test.head(3)

In [ ]:
X_test.to_csv(f"{file_path}/X_test.csv", index=False)
y_test.to_csv(f"{file_path}/y_test.csv", index=False)

### ML Pipeline

Save the complete pipeline.


In [ ]:
joblib.dump(value=classification_pipeline,
            filename=f"{file_path}/classification_pipeline.pkl")

### Feature Importance Plot For LogisticRegression


In [ ]:
df_feature_importance.plot(kind="bar", x="Feature", y="Importance")
plt.savefig(f"{file_path}/features_importance.png", bbox_inches="tight")
plt.title("Feature Importance - LogisticRegression")
plt.show()

print("Feature importance plot saved")